# 2. Agentes con Function Calling Nativo de Groq

## Objetivos de Aprendizaje
- Comprender el mecanismo de "Function Calling" (tool calling) de Groq y sus ventajas.
- Definir herramientas en el formato JSON Schema que requiere la API.
- Implementar un agente que utiliza function calling para interactuar con una API externa (Wikipedia).
- Manejar el flujo de una conversación donde el modelo solicita la ejecución de una función.

## ¿Qué es Function Calling y por qué es mejor?

En el notebook anterior, construimos un agente que funcionaba parseando texto. El LLM escribía su intención de usar una herramienta en un formato específico ("Action: {...}"), y nosotros usábamos expresiones regulares para extraer esa intención. Este método funciona, pero es frágil:

- El LLM puede cometer errores y no generar el texto en el formato exacto.
- El parsing puede fallar si la estructura del texto cambia ligeramente.
- Los argumentos de la función se pasan como un string que debemos convertir a JSON, lo cual puede dar errores.

**Function Calling** es la solución nativa de la API de Groq a este problema. Groq implementa la misma especificación de *tool calling* que OpenAI, así que el formato de las herramientas (JSON Schema) es exactamente el mismo que verás en la documentación de cualquiera de los dos. En lugar de pedirle al modelo que *escriba* qué herramienta quiere usar, le permitimos que nos devuelva una estructura de datos **JSON bien formada** que especifica el nombre de la función y los argumentos que quiere usar. 

**Ventajas:**
1.  **Fiabilidad**: El modelo está entrenado para generar un JSON válido, eliminando casi por completo los errores de formato.
2.  **Seguridad**: Evita la necesidad de ejecutar código que el LLM genera directamente.
3.  **Simplicidad**: No más parsing con expresiones regulares. La intención del modelo es clara y estructurada.

### 1. Instalación y Configuración

In [ ]:
# Instalación de dependencias.
#
# Solo hace falta en Google Colab. En local, `uv sync` ya instaló todo esto con las
# versiones exactas del uv.lock; lanzar pip con -U aquí las actualizaría y rompería
# la reproducibilidad que el curso garantiza a todo el grupo.
import sys

if "google.colab" in sys.modules:
    %pip install -qU langchain-groq groq langgraph requests python-dotenv
else:
    print("Entorno local: las dependencias ya las instaló uv sync.")


In [2]:
import os
import re
import json
import requests
from urllib.parse import quote
from groq import Groq

# Carga de credenciales: funciona igual en Google Colab y en local (.env)
try:
    from google.colab import userdata  # type: ignore
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

assert os.getenv("GROQ_API_KEY"), "Falta GROQ_API_KEY (Colab: Secrets · local: archivo .env)"

# openai/gpt-oss-120b soporta tool calling (function calling) de forma nativa.
# Este notebook usa herramientas (function calling / tool calling) y por eso NO usa
# el modelo grande por defecto: en Groq, openai/gpt-oss-120b falla de forma
# intermitente al generar la llamada a la función (error 400 "tool_use_failed").
# openai/gpt-oss-20b produce el formato correcto de forma fiable.
MODELO = os.getenv("GROQ_MODEL_FAST", "openai/gpt-oss-20b")

WIKIPEDIA_API_URL = "https://es.wikipedia.org/w/api.php"
WIKIPEDIA_SUMMARY_URL = "https://es.wikipedia.org/api/rest_v1/page/summary"
WIKIPEDIA_HEADERS = {
    "User-Agent": "Curso-IA-DUOC/1.0 (notebook educativo; contacto: estudiante@example.com)"
}

# --- Configuración del Cliente Groq ---
# Groq lee GROQ_API_KEY del entorno; no hace falta pasar api_key a mano.
try:
    client = Groq()
    print(f"✅ Cliente Groq configurado correctamente. Modelo: {MODELO}")
except Exception as e:
    print(f"❌ Error configurando el cliente: {e}")
    client = None

✅ Cliente Groq configurado correctamente. Modelo: openai/gpt-oss-20b


### 2. Definición de Herramientas en formato JSON Schema

Primero, definimos la función de Python que queremos que nuestro agente pueda usar. En este caso, una función que busca un resumen en Wikipedia.

Luego, y esto es lo más importante, describimos esa función en un formato de **JSON Schema**. Esta descripción le dice al LLM qué es la herramienta, para qué sirve y qué argumentos necesita.

In [3]:
def _wikipedia_get_json(url, params=None):
    """Solicita JSON a Wikipedia y devuelve un error legible si la respuesta no es válida."""
    try:
        response = requests.get(
            url,
            params=params,
            headers=WIKIPEDIA_HEADERS,
            timeout=10,
        )
        response.raise_for_status()
        return response.json(), None
    except requests.exceptions.JSONDecodeError:
        return None, "Wikipedia devolvió una respuesta vacía o no válida en JSON."
    except requests.exceptions.RequestException as e:
        return None, f"No se pudo consultar Wikipedia: {e}"


def _limitar_oraciones(texto, max_oraciones=2):
    oraciones = re.split(r"(?<=[.!?])\s+", texto.strip())
    return " ".join(oraciones[:max_oraciones])


# La función de Python que realiza la acción
def get_wikipedia_summary(query):
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases."""
    if not query or not query.strip():
        return "No se recibió un término de búsqueda para Wikipedia."

    search_data, error = _wikipedia_get_json(
        WIKIPEDIA_API_URL,
        params={
            "action": "query",
            "list": "search",
            "srsearch": query,
            "srlimit": 1,
            "format": "json",
            "utf8": 1,
        },
    )
    if error:
        return error

    results = search_data.get("query", {}).get("search", [])
    if not results:
        return f"No se encontró ninguna página para '{query}'."

    title = results[0]["title"]
    summary_data, error = _wikipedia_get_json(
        f"{WIKIPEDIA_SUMMARY_URL}/{quote(title)}"
    )
    if error:
        return error

    extract = summary_data.get("extract")
    if not extract:
        return f"Wikipedia no entregó un resumen disponible para '{title}'."

    return _limitar_oraciones(extract, max_oraciones=2)


# Lista de herramientas en formato JSON Schema para la API de Groq
# (Groq usa la misma especificación de tools que OpenAI: el esquema no cambia)
tools_definition = [
    {
        "type": "function",
        "function": {
            "name": "get_wikipedia_summary",
            "description": "Obtiene un resumen conciso de un artículo de Wikipedia para un tema o persona específica.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "El tema o nombre a buscar en Wikipedia. Por ejemplo, 'Albert Einstein'."
                    }
                },
                "required": ["query"]
            }
        }
    }
]

print("✅ Herramienta y su definición para Groq listas.")

✅ Herramienta y su definición para Groq listas.


> ### ⚠️ No todos los modelos hacen function calling igual de bien
>
> El *tool calling* no lo resuelve la API por su cuenta: es el **modelo** el que tiene que
> generar la llamada a la función con el formato exacto que la API espera. Si se equivoca,
> Groq devuelve un error `400` con el código `tool_use_failed`.
>
> Lo medimos con esta misma herramienta, y el resultado es menos intuitivo de lo que parece:
>
> | Vía de código | `openai/gpt-oss-120b` | `openai/gpt-oss-20b` |
> |---|---|---|
> | SDK crudo (esta notebook), prompt en español | 7/10 | **10/10** |
> | Agente de LangChain con prompt del hub, en inglés | **6/6** | 4/6 |
>
> Por eso **esta** notebook usa `openai/gpt-oss-20b`, mientras que la siguiente
> (`3-langchain-agent`) usa el modelo grande: cuál es más fiable **se invierte** según
> cómo se le presenten las herramientas al modelo.
>
> Tres ideas que te llevas a producción:
>
> 1. **El modelo más potente no es automáticamente el mejor para un agente.** Si el agente
>    depende de un formato de salida exacto, la fiabilidad de ese formato pesa tanto como
>    la calidad del razonamiento.
> 2. **Mídelo con tu propio prompt y tus propias herramientas.** Aquí bastó cambiar el prompt
>    del sistema para que cambiara el modelo ganador.
> 3. **`tool_use_failed` es un error esperable**, no un bug: en producción se maneja con
>    reintento.


### 3. El Flujo del Agente con Function Calling

El proceso ahora es más estructurado:

1.  **Usuario -> Agente**: Enviamos la pregunta del usuario y la lista de herramientas (`tools_definition`) al LLM.
2.  **LLM -> Agente**: El LLM analiza la pregunta. Si decide que necesita una herramienta, en lugar de devolver un mensaje de texto, devuelve un objeto `tool_calls`.
3.  **Agente**: Verificamos si la respuesta contiene `tool_calls`. Si es así, ejecutamos la función correspondiente en nuestro código Python.
4.  **Agente -> LLM**: Enviamos el resultado de la función de vuelta al LLM en un nuevo mensaje con `role="tool"`.
5.  **LLM -> Usuario**: El LLM, ahora con la información de la herramienta, genera la respuesta final en lenguaje natural.

In [4]:
def run_agent_with_function_calling(user_query, client, tools_definition):
    if not client:
        return "Cliente no inicializado."

    # Mapeo de nombres de función a las funciones de Python reales
    available_tools = {
        "get_wikipedia_summary": get_wikipedia_summary,
    }

    messages = [{"role": "user", "content": user_query}]
    
    print(f"--- 🚀 Iniciando agente para la consulta: '{user_query}' ---")

    # Primera llamada al modelo
    response = client.chat.completions.create(
        model=MODELO,
        messages=messages,
        tools=tools_definition, # Aquí pasamos la definición de las herramientas
        tool_choice="auto",  # El modelo decide si usar una herramienta o no
    )

    response_message = response.choices[0].message
    messages.append(response_message) # Añadir la respuesta del LLM al historial

    # Comprobar si el modelo quiere llamar a una función
    if response_message.tool_calls:
        print("🤖 El modelo ha decidido usar una herramienta...")
        for tool_call in response_message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            
            print(f"   - Herramienta: {function_name}")
            print(f"   - Argumentos: {function_args}")
            
            # Ejecutar la función
            function_to_call = available_tools[function_name]
            function_response = function_to_call(**function_args)
            
            print(f"   - Resultado: {function_response}")
            
            # Enviar el resultado de vuelta al modelo
            messages.append(
                {
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": function_name,
                    "content": function_response,
                }
            )
        
        # Segunda llamada al modelo, ahora con el resultado de la herramienta
        print("🧠 El modelo está procesando el resultado de la herramienta...")
        second_response = client.chat.completions.create(
            model=MODELO,
            messages=messages
        )
        return second_response.choices[0].message.content
    else:
        # Si el modelo no usó una herramienta, devuelve su respuesta directamente
        print("✅ El modelo ha respondido directamente.")
        return response_message.content

print("✅ Lógica del agente con Function Calling definida.")

✅ Lógica del agente con Function Calling definida.


### 4. Ejecución del Agente

Probemos con una pregunta que claramente necesita conocimiento externo.

In [5]:
query = "¿Quién fue Marie Curie y cuáles fueron sus logros más importantes?"
final_answer = run_agent_with_function_calling(query, client, tools_definition)

print(f"🏁 Respuesta Final del Agente:{final_answer}")

--- 🚀 Iniciando agente para la consulta: '¿Quién fue Marie Curie y cuáles fueron sus logros más importantes?' ---


🤖 El modelo ha decidido usar una herramienta...
   - Herramienta: get_wikipedia_summary
   - Argumentos: {'query': 'Marie Curie'}


   - Resultado: Maria Salomea Skłodowska-Curie, más conocida como Marie Curie o Madame Curie, fue una física y química polaca, luego naturalizada francesa. Pionera en el campo de la radiactividad, es la primera y única persona en recibir dos premios Nobel en distintas especialidades científicas: Física y Química.
🧠 El modelo está procesando el resultado de la herramienta...


🏁 Respuesta Final del Agente:En 1903, junto con sus esposo Pierre Curie y el físico francés Antoine Henri Becquerel, Marie Curie recibió el Premio Nobel de Física por sus investigaciones sobre la radiactividad. En 1911, recibió el Premio Nobel de Química por su trabajo en la identificación de los elementos radio y polonio.


## Conclusiones

El uso de **Function Calling** nativo representa un salto cualitativo en la construcción de agentes. La comunicación entre el LLM y nuestro código es ahora mucho más **robusta, predecible y fácil de depurar**.

Hemos eliminado la necesidad de crear prompts complejos para el formato ReAct y de parsear la salida del modelo. Sin embargo, todavía gestionamos manualmente el estado de la conversación (`messages`) y el flujo de llamadas.

En el próximo notebook, introduciremos **LangChain**, un framework que abstrae esta lógica y nos permite construir agentes aún más potentes con mucho menos código.